# FedShield · NFPC Phase-2 (main result)

Federated mule-account detection on the RBIH NFPC data: ingest → MuleHunter feature families → 3-bank federation (secure aggregation, Byzantine gates, DP) → leakage check → submission.

Attach: the NFPC dataset and the FedShield code dataset. Runtime ≈ 2–3 h on CPU.

In [ ]:
# --- setup: put the FedShield code on the path and point it at the Kaggle inputs ---
import os, sys, glob, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb>=1.1", "numba>=0.60", "xgboost>=2.0"], check=True)
# the repo attached as a Kaggle dataset (any slug): find the folder that contains fedshield/
code = next(p for p in glob.glob("/kaggle/input/*/") + glob.glob("/kaggle/input/*/*/") if os.path.isdir(os.path.join(p, "fedshield")))
sys.path.insert(0, code)
os.environ["FEDSHIELD_WORK"] = "/kaggle/working/work"
os.environ["FEDSHIELD_RESULTS"] = "/kaggle/working/results"
os.environ["FEDSHIELD_DUCKDB_MEM"] = "12GB"        # Kaggle has ~30 GB RAM
print("code:", code)


In [ ]:
os.environ["FEDSHIELD_NFPC_DIR"] = next(p for p in glob.glob("/kaggle/input/*/") if os.path.exists(os.path.join(p, "train_labels.parquet")))
print(os.environ["FEDSHIELD_NFPC_DIR"])

## 1 · Ingest: join transactions with transactions_additional, shard by account

In [ ]:
!cd {code} && python -m fedshield.ingest.nfpc

## 2 · Features: v1 behavioural + MuleHunter families (rolling peaks, channel counts, digit patterns, concurrency, MCC outliers). Batched and resumable

In [ ]:
!cd {code} && python -m fedshield.features.nfpc_v2 && python -m fedshield.features.nfpc_v2 --merge

## 3 · Experiments: local vs FedShield vs centralised, registry, pattern skew, Byzantine, DP, fairness, seeds

In [ ]:
!cd {code} && python -m fedshield.experiments.nfpc full

## 4 · Leakage check: point-in-time features vs full-window features

In [ ]:
!cd {code} && python -m fedshield.experiments.leakage

## 5 · Submission for test_accounts

In [ ]:
!cd {code} && python -m fedshield.experiments.submission
import pandas as pd
pd.read_csv('/kaggle/working/results/submission/submission.csv').head()

In [ ]:
import json
r = json.load(open('/kaggle/working/results/nfpc_full.json'))
print(json.dumps(r['main']['summary'], indent=1))
print(json.dumps(json.load(open('/kaggle/working/results/nfpc_leakage.json')), indent=1)[:2000])